In [ ]:
!python -m spacy download fr_core_news_md

In [64]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from nltk.corpus import stopwords
import nltk
import numpy as np
import spacy 
import re 
from tqdm import tqdm
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters


In [75]:
from sklearn.model_selection import train_test_split


In [65]:

from sklearn.feature_extraction.text import CountVectorizer


L'objectif de ce notebook est d'analyser la vérifiabilité dans le discours politique. Nous avons labellisé 1000 extraits de programmes des législatives de 1990. 

Chaque extrait de texte est associé à 3 variables: 
- le parti politique 
- l'orientation politique (gauche radicale, gauche, centre, droite, droite radicale, indépendants)
- la vérifiabilité/le du contenu du texte: par exemple, un candidat qui affirme qu'il souhaite "augmenter le budget de l'éducation nationale de 15% et d'accroitre le volume horaire d'education morale et civique" sera classé comme vérifiable (label 0) tandis qu'un candidat qui affirme vouloir "donner plus d'importance à l'éducation nationale" sera classé comme non-vérifiable (label 1).

# I- Data exploration

## 1) Divergences d'annotations

On commence par comparer les annotations réalisées par les 3 membres du groupe sur un sous ensemble de 200 textes. On va calculer le Kappa de Fleiss pour s'assurer qu'il n'y a pas trop de désaccords sur la façon dont les labels sont assignés.

In [168]:
df = pd.read_excel("annotations communes.xlsx")

ratings = df[["Annotations JL", "Annotations Henri", "Annotations Julien"]].values

table_comptage, categories = aggregate_raters(ratings)

#computing fleiss kappa score
kappa = fleiss_kappa(table_comptage, method='fleiss')

print(f"Number of texts : {len(df)}")
print(f"Fleiss Kappa : {kappa:.4f}")

Number of texts : 199
Fleiss Kappa : 0.8681


On a un Kappa de Fleiss de 0.87%, ce qui est très robuste et nous assure qu'il n'y a pas trop de divergences d'annotations entre les 3 annotateurs. 

On va aussi comparer la proportion de 0 et de 1 par annotateur sur les 200 annotations communes et sur les datasets annotés spécifiquement par un annotateur:

In [169]:
print("Proportions de 1 chez Henri dans le dataset d'annotations communes:", np.round(df[["Annotations Henri"]].mean(),2))
print("Proportions de 1 chez Julien dans le dataset d'annotations communes:", np.round(df[["Annotations Julien"]].mean(),2))
print("Proportions de 1 chez JL dans le dataset d'annotations communes:", np.round(df[["Annotations JL"]].mean(),2))

Proportions de 1 chez Henri dans le dataset d'annotations communes: Annotations Henri    0.66
dtype: float64
Proportions de 1 chez Julien dans le dataset d'annotations communes: Annotations Julien    0.61
dtype: float64
Proportions de 1 chez JL dans le dataset d'annotations communes: Annotations JL    0.67
dtype: float64


In [170]:
df_henri = pd.read_excel("annotations Henri.xlsx")
df_julien = pd.read_excel("annotations Julien.xlsx")
df_jl = pd.read_excel("annotations JL.xlsx")

print("Proportions de 1 chez Henri dans son dataset d'annotation:", np.round(df_henri[["LDB"]].mean(),2))
print("Proportions de 1 chez Julien dans son dataset d'annotation:", np.round(df_julien[["LDB"]].mean(),2))
print("Proportions de 1 chez JL dans son dataset d'annotation:", np.round(df_jl[["LDB"]].mean(),2))

Proportions de 1 chez Henri dans son dataset d'annotation: LDB    0.74
dtype: float64
Proportions de 1 chez Julien dans son dataset d'annotation: LDB    0.65
dtype: float64
Proportions de 1 chez JL dans son dataset d'annotation: LDB    0.69
dtype: float64


Les proportions sont très stables entre annotateurs, ce qui est satisfaisant et devrait nous rassurer quant au risque de biais dans les annotations.

## 2) Traitement des données

In [171]:
df_henri['source_annotation'] = 'Henri'
df_jl['source_annotation'] = 'JL'
df_julien['source_annotation'] = 'Julien'

df_master = pd.concat([df_henri, df_jl, df_julien], ignore_index=True)

taille_avant = len(df_master)
df_master = df_master.drop_duplicates(subset=['Texte'], keep='first')
taille_apres = len(df_master)

if taille_avant != taille_apres:
    print(f"Attention : {taille_avant - taille_apres} textes doublons supprimés.")

Attention : 9 textes doublons supprimés.


In [172]:
df_master.columns = ['Parti', 'Courant Politique', 'Texte', 'Label', 'source_annotation']

On va maintenant nettoyer le texte et ajouter des features:
- suppression des retours à la lignes, des puces, des tirets de liste
- lemmatisation du texte pour éviter que "franc" et "francs" soient des mots identiques
- détection des entités nommées 
- split en datasets de train et de test et validation 
- sur chaque dataset, on regarde pour chaque classe 0 et 1 quels sont les mots les plus fréquents (pas sur le dataset global pour éviter le data leakage). On ne considère pas l'intersection entre les deux dictionnaires de mots les plus fréquents. 
- on construit des features à partir du texte:
    - ratio de valeurs numériques dans le texte
    - ratio d'entités institutionnles dans le texte 
    - ratio de concretude lexicale: pour un texte donné, c'est le nb de mots fréquents de la classe 0/nb de mots fréquents dans les deux classes 
    - ratio de nominalisation: ratio de mots se finissant par des suffixes d'abstraction de type "-tion, -ité, -isme, -ment"
    - ratio de verbes au futur et au conditionnel / nb total de verbes
    - ratio d'interpellation: nb de pronoms d'interpellation (nous, vous, notre, votre)/ nb total de pronoms
    - ratio d'emphase: nb d'adjectifs + adverbes/nb de verbes 
    - longueur moyenne des phrases 
    - nb d'éléments de ponctuation / nb de phrases 

On définit ci dessous les fonctions qui nous permettent de faire ça:

In [173]:
nlp = spacy.load("fr_core_news_md")

def extraire_toutes_features(texte):
    """
    Nettoie le texte, extrait les entités (NER), génère la lemmatisation,
    et calcule les features stylistiques et morphosyntaxiques.
    """
    # Valeurs par défaut si le texte est vide
    if pd.isna(texte) or str(texte).strip() == "":
        return pd.Series({
            "texte_lemmatise": "", "nb_valeurs_numeriques": 0, 
            "nb_entites_institutionnelles": 0, "densite_ner": 0.0,
            "densite_nominalisation": 0.0, "ratio_futur_cond": 0.0,
            "ratio_interpellation": 0.0, "ratio_emphase": 0.0,
            "mots_par_phrase": 0.0, "densite_ponct_interne": 0.0
        })
    
    # Nettoyage OCR
    texte_clean = re.sub(r'-\s*\n\s*', '', str(texte))
    texte_clean = re.sub(r'[·\-\_\r\n]+', ' ', texte_clean)
    texte_clean = re.sub(r'\s+', ' ', texte_clean).strip()
    
    doc = nlp(texte_clean)
    
    # --- Variables globales d'initialisation ---
    lemmes = []
    longueur_mots = 0
    nb_noms_abstraits = 0
    nb_verbes_total = 0
    nb_verbes_futur_cond = 0
    nb_pronoms_total = 0
    nb_pronoms_interpellation = 0
    nb_adj_adv = 0
    nb_ponct_interne = 0
    
    # Listes de détection sémantique
    mots_interpellation = {"nous", "vous", "notre", "votre", "nos", "vos"}
    suffixes_abstraits = ("tion", "tions", "ité", "ités", "isme", "ismes", "ment", "ments")
    ponctuation_interne = {",", ";", ":"}

    # --- 1. Features Sémantiques (NER) ---
    nb_valeurs_numeriques = sum(1 for token in doc if token.pos_ == "NUM")
    nb_entites_institutionnelles = sum(1 for ent in doc.ents if ent.label_ in ["LOC", "ORG"])
    
    # --- 2. Boucle Morphosyntaxique Principale ---
    for token in doc:
        # Ponctuation interne (Feature E)
        if token.text in ponctuation_interne:
            nb_ponct_interne += 1
            
        # Comptage des vrais mots et lemmatisation
        if not token.is_punct and not token.is_space:
            longueur_mots += 1
            
            if not token.is_stop and len(token.lemma_) > 1:
                lemmes.append(token.lemma_.lower())
                
            # --- Feature A : Nominalisation ---
            if token.pos_ == "NOUN" and token.text.lower().endswith(suffixes_abstraits):
                nb_noms_abstraits += 1
                
            # --- Feature B : Verbes et Temps ---
            if token.pos_ in ["VERB", "AUX"]:
                nb_verbes_total += 1
                # Extraction des traits morphologiques
                tense = token.morph.get("Tense")
                mood = token.morph.get("Mood")
                if "Fut" in tense or "Cnd" in mood:
                    nb_verbes_futur_cond += 1
                    
            # --- Feature C : Interpellation ---
            if token.pos_ in ["PRON", "DET"]:
                nb_pronoms_total += 1
                if token.text.lower() in mots_interpellation:
                    nb_pronoms_interpellation += 1
                    
            # --- Feature D : Emphase ---
            if token.pos_ in ["ADJ", "ADV"]:
                nb_adj_adv += 1

    # --- 3. Découpage en phrases (Feature E) ---
    # Convertir le générateur en liste pour compter les phrases
    nb_phrases = len(list(doc.sents)) 

    # --- Sécurités anti division par zéro ---
    if longueur_mots == 0: longueur_mots = 1
    if nb_verbes_total == 0: nb_verbes_total = 1
    if nb_pronoms_total == 0: nb_pronoms_total = 1
    if nb_phrases == 0: nb_phrases = 1

    # --- Renvoi de la ligne de données ---
    return pd.Series({
        "texte_lemmatise": " ".join(lemmes),
        "nb_valeurs_numeriques": nb_valeurs_numeriques / longueur_mots,
        "nb_entites_institutionnelles": nb_entites_institutionnelles / longueur_mots,
        "densite_nominalisation": nb_noms_abstraits / longueur_mots,
        "ratio_futur_cond": nb_verbes_futur_cond / nb_verbes_total,
        "ratio_interpellation": nb_pronoms_interpellation / nb_pronoms_total,
        "ratio_emphase": nb_adj_adv / nb_verbes_total,
        "mots_par_phrase": longueur_mots / nb_phrases,
        "densite_ponct_interne": nb_ponct_interne / nb_phrases
    })

# --- ETAPE 2 : EXTRACTION DU VOCABULAIRE DISCRIMINANT ---

def extraire_vocabulaire_contrastif(df_train, top_n=30):
    textes_0 = df_train[df_train['Label'] == 0]['texte_lemmatise'].dropna()
    textes_1 = df_train[df_train['Label'] == 1]['texte_lemmatise'].dropna()
    
    vectorizer = CountVectorizer(ngram_range=(1, 2), min_df=2)
    
    def get_top_words(textes):
        matrice = vectorizer.fit_transform(textes)
        somme = matrice.sum(axis=0)
        mots = [(mot, somme[0, idx]) for mot, idx in vectorizer.vocabulary_.items()]
        return set([mot for mot, freq in sorted(mots, key=lambda x: x[1], reverse=True)[:top_n]])
    
    top_mots_0 = get_top_words(textes_0)
    top_mots_1 = get_top_words(textes_1)
    
    intersection = top_mots_0.intersection(top_mots_1)
    return top_mots_0 - intersection, top_mots_1 - intersection,intersection

# --- ETAPE 3 : FABRICATION DES FEATURES LEXICALES ---

def appliquer_features_lexicales(df, vocab_0, vocab_1):
    def compter_mots(texte, vocabulaire):
        if not isinstance(texte, str): return 0
        return sum(1 for mot in texte.split() if mot in vocabulaire)
    
    df['nb_termes_classe_0'] = df['texte_lemmatise'].apply(lambda x: compter_mots(x, vocab_0))
    df['nb_termes_classe_1'] = df['texte_lemmatise'].apply(lambda x: compter_mots(x, vocab_1))
    
    df['ratio_concretude_lexicale'] = df.apply(
        lambda row: row['nb_termes_classe_0'] / (row['nb_termes_classe_0'] + row['nb_termes_classe_1']) 
        if (row['nb_termes_classe_0'] + row['nb_termes_classe_1']) > 0 else 0.5, 
        axis=1
    )
    return df

On peut ensuite appliquer ces fonctions à notre corpus de texte:

In [174]:
tqdm.pandas()
colonnes_extraites = [ "texte_lemmatise",
        "nb_valeurs_numeriques",
        "nb_entites_institutionnelles",
        "densite_nominalisation",
        "ratio_futur_cond",
        "ratio_interpellation",
        "ratio_emphase",
        "mots_par_phrase",
        "densite_ponct_interne"
]
df_master[colonnes_extraites] = df_master['Texte'].progress_apply(extraire_toutes_features)


100%|██████████| 957/957 [00:12<00:00, 79.14it/s] 


In [175]:
df_master[colonnes_extraites]

,texte_lemmatise,nb_valeurs_numeriques,nb_entites_institutionnelles,densite_nominalisation,ratio_futur_cond,ratio_interpellation,ratio_emphase,mots_par_phrase,densite_ponct_interne
0,arrêter accroissement pression fiscal,0.000000,0.000000,0.142857,0.000000,0.000000,1.000000,7.00,0.000000
1,gauche an mitterrand trahir engager ment nom g...,0.012048,0.024096,0.048193,0.000000,0.000000,0.625000,20.75,2.000000
2,bourrer crâne compétitivité crise réalité diri...,0.000000,0.000000,0.089286,0.000000,0.055556,0.500000,56.00,5.000000
3,fermeture hoover bendix montrer patron agir ba...,0.000000,0.000000,0.035714,0.000000,0.142857,1.000000,28.00,2.000000
4,politiciens disent société gérer profit non in...,0.000000,0.016667,0.000000,0.142857,0.200000,1.571429,10.00,0.166667
...,...,...,...,...,...,...,...,...,...
961,france signer convention européen langue minor...,0.000000,0.047619,0.095238,0.000000,0.000000,1.333333,21.00,1.000000
962,reconnaître officiellement langue breton acte ...,0.000000,0.000000,0.041667,0.000000,0.000000,2.500000,12.00,0.000000
963,créer service public breton breton radio télév...,0.000000,0.050000,0.050000,0.000000,0.000000,1.500000,20.00,0.000000
964,ambition alternative france concevoir europe i...,0.000000,0.227273,0.045455,0.000000,0.142857,2.000000,22.00,2.000000


In [176]:
#stratify pour conserver les memes proportions de classe das train et test
df_train, df_test = train_test_split(df_master, test_size=0.2, random_state=42, stratify=df_master['Label'])


In [177]:
vocab_0, vocab_1,intersection = extraire_vocabulaire_contrastif(df_train, top_n=60)

In [178]:
print("vocabularie fréquent dans classe 0",vocab_0)
print("vocabularie fréquent dans classe 1",vocab_1)
print("vocabulaire fréquent dans les deux classes",intersection)

vocabularie fréquent dans classe 0 {'créer', 'activité', 'droit', 'taux', 'plus', 'développer', 'intérêt', 'etat', 'européen', 'rural', 'agriculture', 'loi', 'protection', 'heure', 'quartier', 'retraite', 'impôt', 'défense', 'projet', 'salaire', '000', 'milliard', 'moyen', 'service', 'revenu', 'temps', 'sécurité', 'plan', 'charge', 'million', 'création', 'permettre', 'chômeur', 'logement', 'local', 'formation', 'espace'}
vocabularie fréquent dans classe 1 {'aujourd hui', 'aujourd', '21', 'circonscription', 'solidarité', 'gauche', 'confiance', 'tour', 'ensemble', 'action', 'vouloir', 'besoin', 'monsieur', 'développement', 'socialiste', 'candidat', 'prendre', 'homme', 'donner', 'parti', 'mars', 'citoyen', 'défendre', 'hui', 'avenir', 'bien', 'député', 'monde', 'avoir', 'société', 'économique', 'écologiste', 'communiste', 'progrès', 'force', 'voter', 'jeune'}
vocabulaire fréquent dans les deux classes {'politique', 'travail', 'vie', 'public', 'environnement', 'pays', 'grand', 'être', 'fai

On peut voir ci dessous que la séparation lexicale entre les deux classes est très nette: 
- dans la classe 0, on retrouve des mots:
    - de quantification (000, milliard, million, taux, impôt, salaire, revenu, charge, réduction...),
    - associés à l'espace et à l'administration (ville, territoire, rural, espace, quartier, agriculture, loi, institution, service public, logement),
    - des verbes d'actions (créer, développer, assurer,...)
- dans la classe 1, on retrouve des mots:
    - associés aux éléctions et non au programme du candidat: on a pu observer pendant la phase de labellisation que ces mots sont en général associés aux introductions de programmes, aux conclusions des professions de foi, etc (vote, circonscription, dimanche, 21, mars, 21 mars, tour, candidat, voix, député, voter)
    - associés aux valeurs, et plus généralement à des concepts vagues impossibles à quantifier (solidarité, confiance, solution, démocratie, avenir, valeur, progrès),
    - des verbes modaux d'intention (vouloir, devoir, savoir,..)
- dans les mots fréquents dans les deux classes, on retrouve des themes centraux de la campagne, comme chomage, emploi, pays, etc.

In [165]:
df_train = appliquer_features_lexicales(df_train, vocab_0, vocab_1)
df_test = appliquer_features_lexicales(df_test, vocab_0, vocab_1)


In [166]:
df_train['ratio_concretude_lexicale']

222    0.000000
815    1.000000
887    0.333333
604    0.000000
769    0.000000
         ...   
355    0.333333
425    0.500000
811    0.500000
400    1.000000
683    0.000000
Name: ratio_concretude_lexicale, Length: 765, dtype: float64